# Notebook 06: Cross-Model Scaling Comparison
### openai/gpt-oss-20b vs openai/gpt-oss-120b

This notebook conducts a controlled cross-model comparison on a fixed 500-sample test partition:
- Controlled variables: identical test queries, prompt template (Optimized), retrieved demonstrations, $K=5$, temperature ($0.0$).
- Independent variable: LLM model size (`openai/gpt-oss-20b` vs `openai/gpt-oss-120b`).
- Evaluated dimensions: Accuracy, Macro-F1, P95 Latency, Token consumption, and Cost trade-offs.


In [ ]:
# ==========================================
# 0. Google Colab / Local Environment Setup
# ==========================================
import sys, os
from pathlib import Path

# If running in Google Colab, install repository and dependencies
if "google.colab" in sys.modules:
    print("Detected Google Colab environment. Setting up...")
    !git clone https://github.com/your-username/banking-llm-optimizer.git
    %cd banking-llm-optimizer
    !pip install -r requirements.txt
    
    from google.colab import userdata
    try:
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    except Exception:
        import getpass
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")
else:
    print("Running in local environment.")
    ROOT_DIR = Path(".").resolve()
    if str(ROOT_DIR) not in sys.path:
        sys.path.insert(0, str(ROOT_DIR))


### 1. Initialize Controlled 500-Sample Test Split


In [ ]:
from src.data.loader import BankingDataLoader
from src.pipeline import BankingIntentPipeline
from src.evaluation.metrics import ClassificationMetrics
from src.evaluation.cost_analysis import CostAnalyzer
import pandas as pd
import matplotlib.pyplot as plt

loader = BankingDataLoader()
train_pool, val_df, test_df = loader.load_processed_splits()
test_500 = test_df.head(500).copy()

pipeline = BankingIntentPipeline()
cost_analyzer_20b = CostAnalyzer(input_price_per_million=0.20, output_price_per_million=0.20)
cost_analyzer_120b = CostAnalyzer(input_price_per_million=0.80, output_price_per_million=0.80)


### 2. Benchmark Primary Model: openai/gpt-oss-20b


In [ ]:
print("Evaluating Primary Model: openai/gpt-oss-20b...")
res_20b = pipeline.evaluate_dataset(test_500, strategy="optimized", model="openai/gpt-oss-20b", k=5)
m_20b = ClassificationMetrics.compute_all_metrics(res_20b["true_intent"].tolist(), res_20b["predicted_intent"].tolist())
c_20b = cost_analyzer_20b.summarize_benchmark_run(res_20b["latency_ms"].tolist(), res_20b["input_tokens"].tolist(), res_20b["output_tokens"].tolist())


### 3. Benchmark Secondary Model: openai/gpt-oss-120b


In [ ]:
print("Evaluating Secondary Model: openai/gpt-oss-120b...")
res_120b = pipeline.evaluate_dataset(test_500, strategy="optimized", model="openai/gpt-oss-120b", k=5)
m_120b = ClassificationMetrics.compute_all_metrics(res_120b["true_intent"].tolist(), res_120b["predicted_intent"].tolist())
c_120b = cost_analyzer_120b.summarize_benchmark_run(res_120b["latency_ms"].tolist(), res_120b["input_tokens"].tolist(), res_120b["output_tokens"].tolist())


### 4. Cross-Model Comparison Table and Trade-off Visualizations


In [ ]:
model_comp = [
    {
        "Model": "openai/gpt-oss-20b",
        "Accuracy": round(m_20b["accuracy"], 4),
        "Macro-F1": round(m_20b["macro_f1"], 4),
        "Weighted-F1": round(m_20b["weighted_f1"], 4),
        "P95 Latency (ms)": round(c_20b["latency_p95_ms"], 1),
        "Cost / 1K ($)": round(c_20b["cost_per_1k_queries_usd"], 4)
    },
    {
        "Model": "openai/gpt-oss-120b",
        "Accuracy": round(m_120b["accuracy"], 4),
        "Macro-F1": round(m_120b["macro_f1"], 4),
        "Weighted-F1": round(m_120b["weighted_f1"], 4),
        "P95 Latency (ms)": round(c_120b["latency_p95_ms"], 1),
        "Cost / 1K ($)": round(c_120b["cost_per_1k_queries_usd"], 4)
    }
]
model_comp_df = pd.DataFrame(model_comp)
model_comp_df.to_csv("results/tables/model_comparison.csv", index=False)
display(model_comp_df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.bar(model_comp_df["Model"], model_comp_df["Macro-F1"], color=["#1f77b4", "#2ca02c"])
ax1.set_title("Macro-F1 Comparison")
ax1.set_ylabel("Macro-F1")
ax1.grid(axis="y", linestyle="--", alpha=0.5)

ax2.bar(model_comp_df["Model"], model_comp_df["Cost / 1K ($)"], color=["#ff7f0e", "#d62728"])
ax2.set_title("Cost per 1,000 Queries ($)")
ax2.set_ylabel("USD")
ax2.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("results/figures/model_comparison.png", dpi=300)
plt.show()
